In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import numpy as np

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl
df_fern = df[df["Network ID"] == 11]

df_fern = df_fern.loc[:, df_fern.notna().any()]
df_fern_vanessa = df_fern[df_fern['Phone number'].str.contains("Vanessa", case=False, na=False)]

fern_v_ted = df_fern_vanessa[['Station_name','native_id', 'lat', 'lon', 'Elevation']].reset_index(drop=True)

fern_v_pcds = df_fern_vanessa[['Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
fern_v_pcds['Station ID'] = np.int64(fern_v_pcds['Station ID'])

split_cols = fern_v_pcds['Unique Names'].str.split('->', n=1, expand=True)
# Step 2: assign columns
fern_v_pcds['db_station_name'] = split_cols[0].str.strip()

fern_v_pcds[['Station ID', 'Native ID', 'db_station_name']]
fern_v_ted

,Station_name,native_id,lat,lon,Elevation
0,ThompsonWx,Thompson,54.333115,-126.099445,869
1,BoulderWx,Boulder Creek,55.108173,-127.37474,385
2,BulkleyWx,PGTIS1,53.772183,-122.720729,594
3,ChiefLakeWx,ChiefWx,54.238028,-122.877306,720
4,CPFWx,PGTIS3,53.753775,-122.718864,593
5,CrystalWx,Crystal Lake,54.423494,-122.627844,744
6,Dennis,Dennis,54.78217,-127.514341,895
7,EndakoWx,Endako,54.422392,-125.991293,851
8,GeorgeWx,George,53.883555,-124.710668,859
9,GunnelWx,Gunnel1,58.75475,-123.977683,1460


### manually revise the modification



In [5]:
fern_v_ted

fern_v_ted.loc[fern_v_ted['Station_name'] == 'Dennis', ['lat', 'lon', 'Elevation']] = [54.76159, -127.46765, 895]
fern_v_ted.loc[fern_v_ted['Station_name'] == 'SmitherCmpd', 'Station_name'] = 'SmithersCmpd'

fern_v_ted_update = fern_v_ted
fern_v_ted_update

,Station_name,native_id,lat,lon,Elevation
0,ThompsonWx,Thompson,54.333115,-126.099445,869
1,BoulderWx,Boulder Creek,55.108173,-127.37474,385
2,BulkleyWx,PGTIS1,53.772183,-122.720729,594
3,ChiefLakeWx,ChiefWx,54.238028,-122.877306,720
4,CPFWx,PGTIS3,53.753775,-122.718864,593
5,CrystalWx,Crystal Lake,54.423494,-122.627844,744
6,Dennis,Dennis,54.76159,-127.46765,895
7,EndakoWx,Endako,54.422392,-125.991293,851
8,GeorgeWx,George,53.883555,-124.710668,859
9,GunnelWx,Gunnel1,58.75475,-123.977683,1460


In [6]:
combined_pcds_update = pd.concat([fern_v_pcds[['Station ID', 'db_station_name', 'Native ID']],
                      fern_v_ted_update],
                     axis=1)

combined_pcds_update

combined_pcds_update = combined_pcds_update.rename(columns={
    'Native ID': 'db_native_id',
    'Station ID': 'db_station_id',
    'Station_name': 'updated_station_name',
    'native_id': 'updated_native_id',
    'lat': 'updated_lat',
    'lon': 'updated_lon',
    'Elevation': 'updated_elev'
})
# import os
# print(os.getcwd())

save_path = './output_tables/'
combined_pcds_update.to_csv(save_path + '5_32station_search_Vanessa_pcds_before_after_summary.csv', index=False)
combined_pcds_update

,db_station_id,db_station_name,db_native_id,updated_station_name,updated_native_id,updated_lat,updated_lon,updated_elev
0,1601,ThompsonWx,1163628,ThompsonWx,Thompson,54.333115,-126.099445,869
1,12056,Boulder Creek,BoulderCr,BoulderWx,Boulder Creek,55.108173,-127.37474,385
2,1604,BulkleyWx,1113694,BulkleyWx,PGTIS1,53.772183,-122.720729,594
3,13070,Chief Lake,Chief Lake,ChiefLakeWx,ChiefWx,54.238028,-122.877306,720
4,1606,CPFWx,1113682,CPFWx,PGTIS3,53.753775,-122.718864,593
5,12058,Crystal Lake,CrystalLk,CrystalWx,Crystal Lake,54.423494,-122.627844,744
6,13256,Dennis,Dennis,Dennis,Dennis,54.76159,-127.46765,895
7,1600,EndakoWx,1159701,EndakoWx,Endako,54.422392,-125.991293,851
8,1602,GeorgeWx,1177893,GeorgeWx,George,53.883555,-124.710668,859
9,1595,GunnelWx,1113693,GunnelWx,Gunnel1,58.75475,-123.977683,1460


### Connect to db and update metadata

In [7]:
import sqlalchemy as sa
from sqlalchemy.orm import Session


engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)



for station_name, station_id in zip(combined_pcds_update['db_station_name'], combined_pcds_update['db_station_id']):
    query = sa.text("""
        SELECT DISTINCT m.station_name, m.lat, m.lon, m.elev, s.native_id, s.mod_user
        FROM meta_history AS m
        JOIN meta_station AS s ON m.station_id = s.station_id
        WHERE s.network_id = 11
          AND m.station_name = :station_name
          And m.station_id = :station_id

    """)

    with engine.begin() as conn:  # begin() ensures commit/rollback
        result = conn.execute(query, {"station_name": station_name, "station_id": station_id})
        row = result.first()  # get the first row, None if not found

    if row:
        print(f"Yes, Station '{station_name}' exists in the database.")
    else:
        print(f"No, Station '{station_name}' does NOT exist in the database.")

Yes, Station 'ThompsonWx' exists in the database.
No, Station 'Boulder Creek' does NOT exist in the database.
Yes, Station 'BulkleyWx' exists in the database.
No, Station 'Chief Lake' does NOT exist in the database.
Yes, Station 'CPFWx' exists in the database.
No, Station 'Crystal Lake' does NOT exist in the database.
Yes, Station 'Dennis' exists in the database.
Yes, Station 'EndakoWx' exists in the database.
Yes, Station 'GeorgeWx' exists in the database.
Yes, Station 'GunnelWx' exists in the database.
No, Station 'Hourglass' does NOT exist in the database.
Yes, Station 'Hudson Bay Mtn2' exists in the database.
Yes, Station 'Kluskus' exists in the database.
Yes, Station 'PinkWx' exists in the database.
Yes, Station 'SaxtonWx' exists in the database.
Yes, Station 'Willow-BowronWx' exists in the database.
No, Station 'Bowron Pit' does NOT exist in the database.
No, Station 'Blackhawk' does NOT exist in the database.
No, Station 'Dunster' does NOT exist in the database.
No, Station 'Mac

In [7]:
### Information before update
# import pandas as pd
# import sqlalchemy as sa
# from sqlalchemy.orm import Session

# engine = sa.create_engine(
#     "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca"
#     "&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass",
#     echo=False
# )

# session = Session(engine)

# results_list = []   # store each row as dict

# query = sa.text("""
#     SELECT DISTINCT m.station_id, m.station_name, m.lat, m.lon, m.elev, 
#                     s.native_id, s.mod_user
#     FROM meta_history AS m
#     JOIN meta_station AS s ON m.station_id = s.station_id
#     WHERE s.network_id = 11
#       AND m.station_name = :station_name
#       AND m.station_id = :station_id
# """)

# for station_name, station_id in zip(combined_pcds_update['db_station_name'], combined_pcds_update['db_station_id']):
    
#     with engine.begin() as conn:
#         result = conn.execute(query, {"station_name": station_name, "station_id": station_id})
#         row = result.first()

#     if row:
#         # SQLAlchemy Row → dict
#         results_list.append(dict(row._mapping))
#     else:
#         results_list.append({
#             "station_name": station_name,
#             "lat": None,
#             "lon": None,
#             "elev": None,
#             "native_id": None,
#             "mod_user": None
#         })

# # Convert to DataFrame
# df_results = pd.DataFrame(results_list)

# df_results.to_csv(save_path + '0.0_32station_search_Vanessa_pcds_info_before_update.csv', index=False)
# df_results

### Update

In [8]:
update_meta_history = sa.text("""
    UPDATE meta_history
    SET station_name = :new_name,
        lat = :new_lat,
        lon = :new_lon,
        elev = :new_elev
    WHERE station_id = :station_id
""")

update_meta_station = sa.text("""
    UPDATE meta_station
    SET native_id = :new_native_id
    WHERE station_id = :station_id
""")

with engine.begin() as conn:

    for idx, row in combined_pcds_update.iterrows():

        conn.execute(update_meta_history, {
            "new_name": row["updated_station_name"],
            "new_lat": row["updated_lat"],
            "new_lon": row["updated_lon"],
            "new_elev": row["updated_elev"],
            "station_id": row["db_station_id"]
        })

        conn.execute(update_meta_station, {
            "new_native_id": row["updated_native_id"],
            "station_id": row["db_station_id"]
        })

### Check

In [9]:
query_check = sa.text("""
    SELECT m.station_id, m.station_name, m.lat, m.lon, m.elev,
           s.native_id, s.mod_user
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE m.station_id = ANY(:station_ids)
""")

station_ids = combined_pcds_update['db_station_id'].tolist()

with engine.begin() as conn:
    result = conn.execute(query_check, {"station_ids": station_ids})
    updated_rows = result.fetchall()

# Convert to pandas DataFrame for easy comparison
df_updated = pd.DataFrame([dict(row._mapping) for row in updated_rows])

df_updated.to_csv(save_path + '5_32station_search_Vanessa_pcds_info_after_update.csv', index=False)
